In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import minmax_scale
from sklearn.model_selection import KFold

import tensorflow as tf

import keras
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Flatten, MaxPool2D, Conv2D
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import L1L2, L1, L2
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from tqdm import tqdm
import gc

/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
def draw_feature(H, name = 'Heatmap'):
    plt.clf()
    plt.imshow(H.T, origin='lower120000')
    plt.colorbar()
    plt.title(name)
    plt.show()

In [4]:
def get_binding_feature(feature, bin_size):
  a = np.zeros((bin_size*3, bin_size))
  xc1 = int(bin_size/6)
  xc2 = xc1 + int(bin_size/3)
  xc3 = xc2 + int(bin_size/3)
  xc4 = int(bin_size/4)
  xc5 = xc4 + int(bin_size/2)
  xc6 = xc5 + int((bin_size-xc5)/2)

  yc13 = int(bin_size/4)
  yc45 = int(bin_size/4) + int(bin_size/2)

  a[yc13][xc1] = feature[0]
  a[yc13][xc2] = feature[1]
  a[yc13][xc3] = feature[2]
  a[yc45][xc4] = feature[3]
  a[yc45][xc5] = feature[4]
  a[yc45][xc6] = feature[5]

  a[yc13-1][xc1], a[yc13-1][xc1-1],  a[yc13-1][xc1+1], a[yc13][xc1-1], a[yc13][xc1+1], a[yc13+1][xc1], a[yc13+1][xc1-1],  a[yc13+1][xc1+1]= feature[0], feature[0], feature[0], feature[0], feature[0], feature[0], feature[0], feature[0]
  a[yc13-1][xc2], a[yc13-1][xc2-1],  a[yc13-1][xc2+1], a[yc13][xc2-1], a[yc13][xc2+1], a[yc13+1][xc2], a[yc13+1][xc2-1],  a[yc13+1][xc2+1]= feature[1], feature[1], feature[1], feature[1], feature[1], feature[1], feature[1], feature[1]
  a[yc13-1][xc3], a[yc13-1][xc3-1],  a[yc13-1][xc3+1], a[yc13][xc3-1], a[yc13][xc3+1], a[yc13+1][xc3], a[yc13+1][xc3-1],  a[yc13+1][xc3+1]= feature[2], feature[2], feature[2], feature[2], feature[2], feature[2], feature[2], feature[2]
  a[yc45-1][xc4], a[yc45-1][xc4-1],  a[yc45-1][xc4+1], a[yc45][xc4-1], a[yc45][xc4+1], a[yc45+1][xc4], a[yc45+1][xc4-1],  a[yc45+1][xc4+1]= feature[3], feature[3], feature[3], feature[3], feature[3], feature[3], feature[3], feature[3]
  a[yc45-1][xc5], a[yc45-1][xc5-1],  a[yc45-1][xc5+1], a[yc45][xc5-1], a[yc45][xc5+1], a[yc45+1][xc5], a[yc45+1][xc5-1],  a[yc45+1][xc5+1]= feature[4], feature[4], feature[4], feature[4], feature[4], feature[4], feature[4], feature[4]
  a[yc45-1][xc6], a[yc45-1][xc6-1],  a[yc45-1][xc6+1], a[yc45][xc6-1], a[yc45][xc6+1], a[yc45+1][xc6], a[yc45+1][xc6-1],  a[yc45+1][xc6+1]= feature[5], feature[5], feature[5], feature[5], feature[5], feature[5], feature[5], feature[5]
  a = np.array(a)
  
  return a

In [5]:
# lnc
lnc_doc2vec_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/lnc_doc2vec_dict.txt", header=None, index_col=0)
lnc_ctd_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/lnc_ctd_dict.txt", header=None, index_col=0)
lnc_role2vec_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/lnc_role2vec_dict.txt", header=None, index_col=0)
lnc_kmer_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/lnc_kmer_dict.txt", header=None, index_col=0)
ss_loops_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/Lnc/binding_features_lnc.txt", header=None, index_col=0)

# precursor
mir_doc2vec_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_doc2vec_dict.txt", header=None, index_col=0)
mir_ctd_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_ctd_dict.txt", header=None, index_col=0)
mir_role2vec_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_role2vec_dict.txt", header=None, index_col=0)
mir_kmer_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_kmer_dict.txt", header=None, index_col=0)
ss_mir_loops_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/miRNA Precursor/binding_features_miRNA_precursor.txt", header=None, index_col=0)

# especifico
mir_especifico_doc2vec_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_especifico_doc2vec_dict.txt", header=None, index_col=0)
mir_especifico_ctd_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_especifico_ctd_dict.txt", header=None, index_col=0)
mir_especifico_role2vec_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_especifico_role2vec_dict.txt", header=None, index_col=0)
mir_especifico_kmer_dict = pd.read_csv("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Embedding/Diccionarios/mir_especifico_kmer_dict.txt", header=None, index_col=0)

In [6]:
from tqdm import tqdm

bin_size = 32


negative_pairs_path = "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/Necesarios/txt_interac/negative_pairs.txt"
positive_pairs_path = "/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Datos/Necesarios/txt_interac/mirnas_lncrnas_validated_positive_pairs.txt"

positive_pairs = [[line.strip().split(",")[0],line.strip().split(",")[1]] for line in open(positive_pairs_path,"r").readlines()]
negative_pairs = [[line.strip().split(",")[0],line.strip().split(",")[1]] for line in open(negative_pairs_path,"r").readlines()]

all_pairs = positive_pairs + negative_pairs
rows = []

for pair in tqdm(all_pairs):
    lnc = pair[0]
    mir = pair[1]
    lnc_vec = lnc_kmer_dict.loc[lnc].tolist()
    mir_vec = mir_kmer_dict.loc[mir].tolist()
    mir_especifico_vec = mir_especifico_kmer_dict.loc[mir].tolist()
    assert len(lnc_vec) == len(mir_vec) == len(mir_especifico_vec), f'Vector lengths are inconsistent ctd {len(lnc_vec)} {len(mir_vec)} {len(mir_especifico_vec)}'
    H1, xedges, yedges = np.histogram2d(lnc_vec, mir_vec, bins=bin_size, density=True)
    H2, xedges, yedges = np.histogram2d(lnc_vec, mir_especifico_vec, bins=bin_size, density=True)
    H3, xedges, yedges = np.histogram2d(mir_especifico_vec, mir_vec, bins=bin_size, density=True)
    H = np.vstack((H1, H2, H3))
    H = minmax_scale(H)
    H_kmer = H[:, :, np.newaxis]
    
    lnc_vec = lnc_doc2vec_dict.loc[lnc].tolist()
    mir_vec = mir_doc2vec_dict.loc[mir].tolist()
    mir_especifico_vec = mir_especifico_doc2vec_dict.loc[mir].tolist()
    H1, xedges, yedges = np.histogram2d(lnc_vec, mir_vec, bins=bin_size, density=True)
    H2, xedges, yedges = np.histogram2d(lnc_vec, mir_especifico_vec, bins=bin_size, density=True)
    H3, xedges, yedges = np.histogram2d(mir_especifico_vec, mir_vec, bins=bin_size, density=True)
    H = np.vstack((H1, H2, H3))
    H = minmax_scale(H)
    H_doc2vec = H[:, :, np.newaxis]

    lnc_vec = lnc_ctd_dict.loc[lnc].tolist()
    mir_vec = mir_ctd_dict.loc[mir].tolist()
    mir_especifico_vec = mir_especifico_ctd_dict.loc[mir].tolist()
    H1, xedges, yedges = np.histogram2d(lnc_vec, mir_vec, bins=bin_size, density=True)
    H2, xedges, yedges = np.histogram2d(lnc_vec, mir_especifico_vec, bins=bin_size, density=True)
    H3, xedges, yedges = np.histogram2d(mir_especifico_vec, mir_vec, bins=bin_size, density=True)
    H = np.vstack((H1, H2, H3))
    H = minmax_scale(H)
    H_ctd = H[:, :, np.newaxis]

    lnc_vec = lnc_role2vec_dict.loc[lnc].tolist()
    mir_vec = mir_role2vec_dict.loc[mir].tolist()
    mir_especifico_vec = mir_especifico_role2vec_dict.loc[mir].tolist()
    H1, xedges, yedges = np.histogram2d(lnc_vec, mir_vec, bins=bin_size, density=True)
    H2, xedges, yedges = np.histogram2d(lnc_vec, mir_especifico_vec, bins=bin_size, density=True)
    H3, xedges, yedges = np.histogram2d(mir_especifico_vec, mir_vec, bins=bin_size, density=True)
    H = np.vstack((H1, H2, H3))
    H = minmax_scale(H)
    H_role2vec = H[:, :, np.newaxis]
    
    feature_list = ss_loops_dict.loc[f'{lnc}_{mir}'].tolist()
    H = get_binding_feature(feature_list, bin_size)
    H_binding_feature_lnc = H[:, :, np.newaxis]

    feature_list = ss_mir_loops_dict.loc[f'{lnc}_{mir}'].tolist()
    H = get_binding_feature(feature_list, bin_size)
    H_binding_feature_mir = H[:, :, np.newaxis]

    mat = np.concatenate([H_kmer,H_doc2vec,H_ctd,H_role2vec, H_binding_feature_lnc, H_binding_feature_mir], axis=2)
    rows.append(mat)
    
data = np.array(rows)
np.save("/Users/andrescubillovillalobos/Documents/CompGen_Inc/CNN_lnc_mIR/Red Convolucional/data2D.npy", data)

100%|██████████| 45977/45977 [01:02<00:00, 730.29it/s]
